# ISIC 2019 — ResNet50, SVM và thử nghiệm loss

Mục tiêu: đánh giá việc thay bộ phân loại bằng SVM và thay loss,
trong khi giữ nguyên tiền xử lý resize và split của baseline.

- B00: ResNet50 + CrossEntropy thường, đã huấn luyện và khóa.
- SVM: dùng đặc trưng ResNet50 B00 đóng băng để học Linear SVM.
- L01: ResNet50 + weighted CrossEntropy, huấn luyện độc lập.

Chọn cấu hình bằng validation macro-F1; chưa dùng test để lựa chọn.

## 1. Import thư viện

In [1]:
from pathlib import Path
import os, json, random, hashlib, shutil, time, zipfile
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import cv2
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision import models, transforms
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score, balanced_accuracy_score
from IPython.display import display

cv2.setNumThreads(1)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)


Device: cuda


## 2. Cấu hình cố định — khóa trước thí nghiệm
Không thay loss, optimizer, augmentation, seed hoặc split riêng cho A1. Nếu cần đổi, tạo một phiên bản thí nghiệm mới.

In [2]:
INPUT_ROOT = Path('/kaggle/input')
DATA_ROOT = None  # Nếu có nhiều CSV trùng tên: đặt đường dẫn dataset ISIC cụ thể.
IMAGE_ROOT = None  # Mặc định quét trong dataset chứa GroundTruth.
OUT = Path('/kaggle/working/ISIC2019_R50_CE_701020_v1')
# Tìm Output baseline đã gắn trong Input.
baseline_folders = [
    p.parent
    for p in INPUT_ROOT.rglob("split_manifest.csv")
    if (p.parent / "B00_R50" / "complete.json").is_file()
    and (p.parent / "B00_R50" / "best_resnet50.pt").is_file()
]

assert len(baseline_folders) == 1, (
    f"Cần đúng một Output baseline đầy đủ; "
    f"tìm thấy {len(baseline_folders)}."
)

RESUME_ROOT = str(baseline_folders[0])
print("Dùng lại baseline:", RESUME_ROOT)
A1_REVIEWED = False
A1_REVIEW_NOTE = ''  # Ghi các ID đã xem + nhận xét bờ, màu, chi tiết bị sửa.
RUN_TEST = False
METHOD_FROZEN_NOTE = ''

LABELS = ['MEL', 'NV', 'BCC', 'AK', 'BKL', 'DF', 'VASC', 'SCC']
SEED = 42
IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 16
PATIENCE = 4
NUM_WORKERS = 2
LR = 1e-4
WEIGHT_DECAY = 1e-4
HAIR = dict(kernel=9, threshold=20, min_area=8, max_area=500,
            min_length=12, min_elongation=4.0, max_width=5.0,
            max_fraction=0.03, radius=2)
OUT.mkdir(parents=True, exist_ok=True)

def sha_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

def dump_json(path, data):
    Path(path).write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding='utf-8')

def reset_seed():
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

reset_seed()


Dùng lại baseline: /kaggle/input/notebooks/tuanducnguyen123/isic2019-a1-v3-quality-check-kh-l-ng/ISIC2019_R50_CE_701020_v1


## 3. Chọn baseline và thư mục kết quả

Tái sử dụng checkpoint và split của B00 ResNet50 đã khóa.
Kết quả SVM và loss mới được lưu riêng; chưa sử dụng test.

In [3]:
## Cell S1 — Chọn B00 và Output

# S1 — chạy ngay sau mục 1–2 cũ, trước mục 3–4 cũ.
BASELINE_INPUT = None  # Điền thư mục B00 nếu có nhiều Output cùng loại.
if BASELINE_INPUT is None:
    found=[p.parent for p in INPUT_ROOT.rglob('protocol_lock.json')
           if (p.parent/'B00_R50/complete.json').is_file()
           and (p.parent/'B00_R50/best_resnet50.pt').is_file()]
    assert len(found)==1,f'Cần 1 Output baseline đầy đủ, tìm thấy {found}'
    BASELINE_INPUT=str(found[0])
B00_ROOT=Path(BASELINE_INPUT)
RESUME_ROOT=str(B00_ROOT)
A1_REVIEWED=False
RUN_TEST=False
NEXT_OUT=Path('/kaggle/working/ISIC2019_R50_SVM_Loss_v1')
NEXT_OUT.mkdir(parents=True,exist_ok=True)
# Nếu khôi phục từ Output đã lưu: trỏ đến thư mục ISIC2019_R50_SVM_Loss_v1.
RESTORE_NEXT_ROOT=None
if RESTORE_NEXT_ROOT:
    src=Path(RESTORE_NEXT_ROOT)
    assert (src/'reference.json').is_file()
    for p in src.rglob('*'):
        if p.is_file():
            dst=NEXT_OUT/p.relative_to(src);dst.parent.mkdir(parents=True,exist_ok=True)
            if dst.exists():assert sha_file(dst)==sha_file(p),f'File khác: {dst}'
            else:shutil.copy2(p,dst)
print('B00:',B00_ROOT,'\nOutput mới:',NEXT_OUT)


B00: /kaggle/input/notebooks/tuanducnguyen123/isic2019-a1-v3-quality-check-kh-l-ng/ISIC2019_R50_CE_701020_v1 
Output mới: /kaggle/working/ISIC2019_R50_SVM_Loss_v1


## 4. Đọc nhãn và kiểm tra dữ liệu
Nhãn lấy từ GroundTruth, không lấy từ tên thư mục. SHA-256 chỉ phát hiện ảnh giống byte; không phát hiện mọi ảnh gần trùng. Các nhóm chứa ảnh giống byte nhưng nhãn mâu thuẫn được loại và lưu báo cáo, không tự đổi nhãn.

In [4]:
root = Path(DATA_ROOT) if DATA_ROOT else INPUT_ROOT

def unique_file(name):
    found = list(root.rglob(name))
    if len(found) != 1:
        raise ValueError(f'Cần đúng 1 {name}; tìm thấy {len(found)}. Đặt DATA_ROOT.')
    return found[0]

truth_path = unique_file('ISIC_2019_Training_GroundTruth.csv')
meta_path = unique_file('ISIC_2019_Training_Metadata.csv')
truth = pd.read_csv(truth_path)
meta = pd.read_csv(meta_path)
assert truth.image.is_unique and meta.image.is_unique
assert len(truth) == 25331, 'Cần đủ bộ train ISIC 2019 chính thức.'
y = truth[LABELS].apply(pd.to_numeric, errors='raise').to_numpy()
assert np.isin(y, [0, 1]).all() and (y.sum(1) == 1).all()
if 'UNK' in truth:
    assert pd.to_numeric(truth.UNK).eq(0).all()
frame = truth[['image']].copy()
frame['label'] = y.argmax(1)
frame['label_name'] = [LABELS[i] for i in frame.label]
frame = frame.merge(meta[['image', 'lesion_id']], on='image', how='left', validate='one_to_one', indicator=True)
assert frame['_merge'].eq('both').all()
frame = frame.drop(columns='_merge')
missing = frame.lesion_id.isna() | frame.lesion_id.astype(str).str.strip().eq('')
frame['group'] = 'LESION_' + frame.lesion_id.astype(str)
frame.loc[missing, 'group'] = 'MISSING_' + frame.loc[missing, 'image']

scan_root = Path(IMAGE_ROOT) if IMAGE_ROOT else truth_path.parent
wanted = set(frame.image)
image_map = {}
for p in scan_root.rglob('*'):
    if p.suffix.lower() in {'.jpg', '.jpeg', '.png'} and p.stem in wanted:
        if p.stem in image_map:
            raise ValueError(f'Trùng đường dẫn image ID {p.stem}; đặt IMAGE_ROOT chính xác.')
        image_map[p.stem] = str(p)
frame['path'] = frame.image.map(image_map)
assert frame.path.notna().all(), 'Thiếu ảnh: kiểm tra IMAGE_ROOT.'
with ThreadPoolExecutor(max_workers=4) as pool:
    frame['file_sha256'] = list(pool.map(sha_file, frame.path))
conflict_hashes = frame.groupby('file_sha256').label.nunique()
bad = frame.file_sha256.isin(conflict_hashes[conflict_hashes > 1].index)
frame[bad].to_csv(OUT / 'conflicting_exact_duplicates.csv', index=False)
removed = int(bad.sum())
frame = frame[~bad].copy().reset_index(drop=True)

# Union-find: cùng lesion hoặc cùng bytes phải nằm cùng tập.
parent = {g: g for g in frame.group.unique()}
def find(g):
    while parent[g] != g:
        parent[g] = parent[parent[g]]
        g = parent[g]
    return g
for _, block in frame.groupby('file_sha256'):
    groups = block.group.unique()
    for g in groups[1:]:
        a, b = find(groups[0]), find(g)
        if a != b:
            parent[max(a, b)] = min(a, b)
frame['group'] = frame.group.map(find)
assert frame.groupby('group').label.nunique().max() == 1, 'Một nhóm có nhãn mâu thuẫn; cần rà soát.'
frame = frame.sort_values('image').reset_index(drop=True)
dump_json(OUT / 'data_audit.json', dict(original=len(truth), retained=len(frame),
    excluded_conflicting_images=removed, missing_lesion_id=int(missing.sum()),
    exact_duplicate_excess=int(frame.file_sha256.duplicated().sum()),
    groundtruth_sha=sha_file(truth_path), metadata_sha=sha_file(meta_path)))
print('Giữ:', len(frame), '| loại ảnh mâu thuẫn:', removed, '| thiếu lesion_id:', int(missing.sum()))


Giữ: 25327 | loại ảnh mâu thuẫn: 4 | thiếu lesion_id: 2084


## 5. Split 70/10/20 theo nhóm và lưu cố định
Dùng 10 fold phân tầng theo nhóm: fold 0–1 test, fold 2 validation, fold 3–9 train. Tỉ lệ ảnh xấp xỉ 70/10/20 do không tách một tổn thương sang hai tập. Không tìm seed/split theo điểm mô hình. Khi chạy lại hoặc tiếp tục phiên, tái sử dụng manifest đã lưu.

In [5]:
manifest_path = OUT / 'split_manifest.csv'
resume = Path(RESUME_ROOT) if RESUME_ROOT else None
if not manifest_path.exists() and resume:
    shutil.copy2(resume / 'split_manifest.csv', manifest_path)
if manifest_path.exists():
    old = pd.read_csv(manifest_path, keep_default_na=False).sort_values('image').reset_index(drop=True)
    for col in ['image', 'label_name', 'group', 'file_sha256']:
        assert old[col].equals(frame[col].reset_index(drop=True)), f'Manifest khác dữ liệu: {col}'
    frame['split'] = old['split']
else:
    folds = np.full(len(frame), -1)
    splitter = StratifiedGroupKFold(n_splits=10, shuffle=True, random_state=SEED)
    for fold, (_, idx) in enumerate(splitter.split(frame, frame.label, frame.group)):
        folds[idx] = fold
    frame['split'] = np.where(folds < 2, 'test', np.where(folds == 2, 'val', 'train'))
    frame.drop(columns='path').to_csv(manifest_path, index=False)
assert set(frame.split) == {'train', 'val', 'test'}
for column in ['image', 'group', 'file_sha256']:
    assert frame.groupby(column).split.nunique().max() == 1, f'Leakage: {column}'
parts = {s: frame[frame.split == s].reset_index(drop=True) for s in ['train', 'val', 'test']}
for s, data in parts.items():
    assert set(data.label_name) == set(LABELS), f'{s} thiếu lớp.'
    data.drop(columns='path').to_csv(OUT / f'{s}_manifest.csv', index=False)
distribution = pd.crosstab(frame.label_name, frame.split).reindex(LABELS)[['train', 'val', 'test']]
distribution.to_csv(OUT / 'class_distribution.csv')
display(distribution)
print(frame.split.value_counts(), '\nTỉ lệ:', frame.split.value_counts(normalize=True).round(4).to_dict())


split,train,val,test
label_name,,,
MEL,3077,478,965
NV,9019,1247,2607
BCC,2374,348,601
AK,611,84,172
BKL,1847,285,492
DF,187,20,32
VASC,151,22,80
SCC,457,55,116


split
train    17723
test      5065
val       2539
Name: count, dtype: int64 
Tỉ lệ: {'train': 0.6998, 'test': 0.2, 'val': 0.1002}


## 6. Tiền xử lý B00 và A1
B00 ép ảnh về 224×224, có thể méo ảnh chữ nhật. A1 chỉ thêm khử lông sau cùng phép resize. Black-hat tìm vùng tối nhỏ; lọc thành phần dài/mảnh bằng hình học; Telea nội suy vùng mask. Không phải DullRazor nguyên bản, không phải mask y khoa. Bộ lọc có thể bỏ sót lông cong/giao nhau hoặc nhầm cấu trúc tổn thương; giới hạn 3% diện tích không đảm bảo an toàn.

In [6]:
def preprocess(image, method):
    image = image.convert('RGB').resize((IMG_SIZE, IMG_SIZE), Image.Resampling.BILINEAR)
    rgb = np.asarray(image).copy()
    mask = np.zeros(rgb.shape[:2], dtype=np.uint8)
    if method == 'B00_R50':
        return image, mask, False
    assert method == 'A1_R50'
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (HAIR['kernel'], HAIR['kernel']))
    response = cv2.morphologyEx(gray, cv2.MORPH_BLACKHAT, kernel)
    raw = (response > HAIR['threshold']).astype(np.uint8)
    count, components, stats, _ = cv2.connectedComponentsWithStats(raw, 8)
    for k in range(1, count):
        area = stats[k, cv2.CC_STAT_AREA]
        if not HAIR['min_area'] <= area <= HAIR['max_area']:
            continue
        yy, xx = np.where(components == k)
        points = np.column_stack([xx, yy]).astype(np.float32)
        (_, _), (w, h), _ = cv2.minAreaRect(points)
        length, width = max(w, h) + 1, min(w, h) + 1
        if length >= HAIR['min_length'] and width <= HAIR['max_width'] and length / width >= HAIR['min_elongation']:
            mask[components == k] = 255
    if np.any(mask):
        mask = cv2.dilate(mask, np.ones((3, 3), np.uint8), iterations=1)
    fallback = float((mask > 0).mean()) > HAIR['max_fraction']
    if fallback or not np.any(mask):
        return image, mask, fallback
    repaired = cv2.inpaint(rgb, mask, HAIR['radius'], cv2.INPAINT_TELEA)
    return Image.fromarray(repaired), mask, fallback


## 7. Dataset và DataLoader
Chuẩn hóa ImageNet là một phần cố định của cả hai lượt. Không WeightedRandomSampler, không class weights, không label smoothing.

In [7]:
normalize = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])
class SkinDataset(Dataset):
    def __init__(self, data, method):
        self.data = data.reset_index(drop=True)
        self.method = method
    def __len__(self):
        return len(self.data)
    def __getitem__(self, index):
        row = self.data.iloc[index]
        with Image.open(row.path) as im:
            image, _, _ = preprocess(im, self.method)
        return normalize(image), int(row.label)

def seed_worker(worker_id):
    seed = torch.initial_seed() % (2**32)
    random.seed(seed); np.random.seed(seed)

def make_loader(data, method, train=False):
    generator = torch.Generator().manual_seed(SEED)
    return DataLoader(SkinDataset(data, method), batch_size=BATCH_SIZE,
        shuffle=train, num_workers=NUM_WORKERS, pin_memory=True,
        worker_init_fn=seed_worker, generator=generator, drop_last=False)


## 8. ResNet50, Cross-Entropy và hàm đánh giá
Hai mô hình bắt đầu từ cùng trọng số ImageNet và cùng seed cho lớp cuối; không dùng checkpoint B00 để train A1.

In [8]:
def build_model():
    model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
    model.fc = nn.Linear(model.fc.in_features, len(LABELS))
    return model.to(DEVICE)

criterion = nn.CrossEntropyLoss()  # Không weight; label_smoothing mặc định 0.

def metrics(y, pred):
    return dict(accuracy=float(accuracy_score(y, pred)),
        macro_f1=float(f1_score(y, pred, labels=range(8), average='macro', zero_division=0)),
        balanced_accuracy=float(balanced_accuracy_score(y, pred)))

def evaluate(model, loader):
    model.eval()
    loss_sum, total, labels, probabilities = 0.0, 0, [], []
    with torch.no_grad():
        for images, y in loader:
            images, y = images.to(DEVICE), y.to(DEVICE)
            logits = model(images)
            loss_sum += criterion(logits, y).item() * len(y)
            total += len(y)
            labels.extend(y.cpu().tolist())
            probabilities.append(logits.softmax(1).cpu().numpy())
    probabilities = np.concatenate(probabilities)
    pred = probabilities.argmax(1)
    return loss_sum / total, metrics(labels, pred), np.array(labels), pred, probabilities

def export_evaluation(model, data, method, folder, split):
    loss, scores, y, pred, probs = evaluate(model, make_loader(data, method))
    result = data[['image', 'group', 'file_sha256', 'label_name']].rename(columns={'label_name':'true_label'}).copy()
    result['pred_label'] = [LABELS[i] for i in pred]
    for i, label in enumerate(LABELS): result['p_' + label] = probs[:, i]
    result.to_csv(folder / f'{split}_predictions.csv', index=False)
    report = classification_report(y, pred, labels=range(8), target_names=LABELS, output_dict=True, zero_division=0)
    pd.DataFrame(report).T.to_csv(folder / f'{split}_per_class.csv')
    cm = confusion_matrix(y, pred, labels=range(8))
    pd.DataFrame(cm, index=LABELS, columns=LABELS).to_csv(folder / f'{split}_confusion.csv')
    fig, ax = plt.subplots(figsize=(7, 6)); ax.imshow(cm, cmap='Blues')
    ax.set(xticks=range(8), yticks=range(8), xticklabels=LABELS, yticklabels=LABELS,
           xlabel='Dự đoán', ylabel='Nhãn thật', title=f'{method} — {split}')
    for i in range(8):
        for j in range(8): ax.text(j, i, str(cm[i,j]), ha='center', va='center', color='white' if cm[i,j] > cm.max()/2 else 'black', fontsize=8)
    fig.tight_layout(); fig.savefig(folder / f'{split}_confusion.png', dpi=140); plt.close(fig)
    dump_json(folder / f'{split}_metrics.json', dict(loss=loss, **scores))
    return scores


## 9. Hàm train dùng chung — checkpoint theo validation macro-F1

In [9]:
def train_experiment(method):
    assert DEVICE.type == 'cuda', 'Bật GPU; không train ResNet50 trên CPU.'
    folder = OUT / method
    if not folder.exists() and resume and (resume / method / 'complete.json').exists():
        shutil.copytree(resume / method, folder)
    folder.mkdir(exist_ok=True)
    complete_path = folder / 'complete.json'
    if complete_path.exists():
        saved = json.loads(complete_path.read_text())
        assert saved['lock_sha'] == LOCK_SHA
        assert sha_file(folder / 'best_resnet50.pt') == saved['checkpoint_sha']
        print(method, 'đã hoàn thành; tái sử dụng kết quả, không train lại.')
        return saved
    if (folder / 'history.csv').exists():
        raise RuntimeError('Lượt train trước chưa hoàn thành. Giữ output đó; dùng OUT mới để chạy lại cả protocol, không âm thầm resume optimizer.')
    reset_seed()
    model = build_model()
    # Hash khởi tạo: xác minh hai lượt khởi tạo giống nhau.
    initial = hashlib.sha256()
    for key, value in model.state_dict().items():
        initial.update(key.encode()); initial.update(value.detach().cpu().numpy().tobytes())
    init_sha = initial.hexdigest()
    train_loader = make_loader(parts['train'], method, train=True)
    val_loader = make_loader(parts['val'], method)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    best, wait, history = -1.0, 0, []
    started = time.perf_counter()
    for epoch in range(1, EPOCHS + 1):
        model.train(); loss_sum = 0.0; total = 0
        for images, y in train_loader:
            images, y = images.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            logits = model(images)
            loss = criterion(logits, y)
            loss.backward(); optimizer.step()
            loss_sum += loss.item() * len(y); total += len(y)
        val_loss, scores, _, _, _ = evaluate(model, val_loader)
        history.append(dict(epoch=epoch, train_loss=loss_sum/total, val_loss=val_loss,
                            lr=optimizer.param_groups[0]['lr'], **scores))
        pd.DataFrame(history).to_csv(folder / 'history.csv', index=False)
        print(f'{method} epoch {epoch:02d}: train={loss_sum/total:.4f}, val={val_loss:.4f}, F1={scores["macro_f1"]:.4f}')
        if scores['macro_f1'] > best + 1e-5:
            best, best_epoch, wait = scores['macro_f1'], epoch, 0
            torch.save(model.state_dict(), folder / 'best_resnet50.pt')
        else:
            wait += 1
        scheduler.step()
        if wait >= PATIENCE: break
    model.load_state_dict(torch.load(folder / 'best_resnet50.pt', map_location=DEVICE, weights_only=True))
    scores = export_evaluation(model, parts['val'], method, folder, 'val')
    h = pd.DataFrame(history)
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].plot(h.epoch, h.train_loss, label='train'); axes[0].plot(h.epoch, h.val_loss, label='val'); axes[0].legend()
    axes[1].plot(h.epoch, h.macro_f1); axes[1].set_title('Validation macro-F1')
    for ax in axes: ax.set_xlabel('Epoch'); ax.grid(alpha=.2)
    fig.tight_layout(); fig.savefig(folder / 'learning_curves.png', dpi=120); plt.close(fig)
    saved = dict(method=method, lock_sha=LOCK_SHA, initial_sha=init_sha,
        checkpoint_sha=sha_file(folder / 'best_resnet50.pt'), best_epoch=best_epoch,
        completed_epochs=len(history), seconds=time.perf_counter()-started, **scores)
    dump_json(complete_path, saved)
    del model, optimizer, train_loader, val_loader
    torch.cuda.empty_cache()
    return saved


## 10. Kiểm tra điều kiện so sánh

Đối chiếu mã ảnh, nhãn, nhóm tổn thương, cấu hình và checkpoint
với B00. Chuẩn bị hàm đánh giá trên cùng tập validation.

In [10]:
# S2 — sau khi chạy mục 3,4,5,8,9,10 cũ. Không chạy mục train B00/A1 cũ.
import inspect, warnings, joblib
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
from sklearn.exceptions import ConvergenceWarning
from threadpoolctl import threadpool_limits

b00_config=json.loads((B00_ROOT/'protocol_lock.json').read_text())
b00_result=json.loads((B00_ROOT/'B00_R50/complete.json').read_text())
assert sha_file(B00_ROOT/'protocol_lock.json')==b00_result['lock_sha']
assert sha_file(B00_ROOT/'B00_R50/best_resnet50.pt')==b00_result['checkpoint_sha']
assert sha_file(B00_ROOT/'split_manifest.csv')==b00_config['split_sha']
assert RUN_TEST is False and A1_REVIEWED is False
cols=['image','label','label_name','group','file_sha256','split']
ref=pd.read_csv(B00_ROOT/'split_manifest.csv',keep_default_na=False)
for name,actual,expected in [('all',frame,ref)]+[(s,parts[s],ref[ref.split==s]) for s in ['train','val','test']]:
    pd.testing.assert_frame_equal(actual[cols].sort_values('image').reset_index(drop=True).astype(str),
        expected[cols].sort_values('image').reset_index(drop=True).astype(str))
for c in ['image','group','file_sha256']:
    assert frame.groupby(c).split.nunique().max()==1,f'Leakage {c}'
for k,v in dict(labels=LABELS,seed=SEED,size=IMG_SIZE,batch=BATCH_SIZE,epochs=EPOCHS,
        patience=PATIENCE,workers=NUM_WORKERS,lr=LR,weight_decay=WEIGHT_DECAY).items():
    assert v==b00_config[k],f'Khác B00: {k}'
assert isinstance(criterion,nn.CrossEntropyLoss) and criterion.weight is None
assert criterion.label_smoothing==0 and criterion.reduction=='mean'
assert len(normalize.transforms)==2
assert isinstance(normalize.transforms[0],transforms.ToTensor)
assert isinstance(normalize.transforms[1],transforms.Normalize)
assert list(normalize.transforms[1].mean)==[0.485,0.456,0.406]
assert list(normalize.transforms[1].std)==[0.229,0.224,0.225]
shared_source='\n'.join(inspect.getsource(f) for f in [reset_seed,SkinDataset.__init__,
    SkinDataset.__getitem__,SkinDataset.__len__,seed_worker,make_loader,build_model,
    metrics,evaluate,export_evaluation,train_experiment])
source_hash=hashlib.sha256(shared_source.encode()).hexdigest()
assert source_hash==(B00_ROOT/'training_source.sha256').read_text().strip(),'Code train khác B00'
assert hashlib.sha256(inspect.getsource(preprocess).encode()).hexdigest()==b00_config['preprocessing_source'],'preprocess cũ đã bị thay'
env=dict(torch=str(torch.__version__),torchvision=str(torchvision.__version__),numpy=np.__version__,
    pandas=pd.__version__,sklearn=__import__('sklearn').__version__,cv2=cv2.__version__,
    pillow=__import__('PIL').__version__,gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
# Cho phép dùng CPU đọc cache và fit SVM; không âm thầm đổi phiên bản thư viện.
for k in env:
    if k!='gpu':assert env[k]==b00_config['environment'][k],f'Môi trường khác B00: {k}={env[k]}'

def next_lock(path,obj):
    path=Path(path)
    if path.exists():assert json.loads(path.read_text())==obj,f'Lock khác: {path}'
    else:dump_json(path,obj)

reference=dict(baseline_lock_sha=b00_result['lock_sha'],checkpoint_sha=b00_result['checkpoint_sha'],
    split_sha=b00_config['split_sha'],training_source_sha=source_hash,labels=LABELS,
    classifier_test_used=False)
next_lock(NEXT_OUT/'reference.json',reference)
base_predictions=pd.read_csv(B00_ROOT/'B00_R50/val_predictions.csv')
assert base_predictions.image.is_unique
truth=parts['val'][['image','group','file_sha256','label_name']].rename(columns={'label_name':'true_label'})
assert set(base_predictions.image)==set(truth.image)
checked=base_predictions.merge(truth,on='image',suffixes=('','_manifest'),validate='one_to_one')
for c in ['group','file_sha256','true_label']:assert checked[c].eq(checked[c+'_manifest']).all()
label_to_id={l:i for i,l in enumerate(LABELS)}
base_scores=metrics(base_predictions.true_label.map(label_to_id),base_predictions.pred_label.map(label_to_id))
for k,v in base_scores.items():assert np.isclose(v,b00_result[k],atol=1e-10,rtol=0)
shutil.copy2(B00_ROOT/'protocol_lock.json',NEXT_OUT/'B00_protocol_lock.json')
shutil.copy2(B00_ROOT/'B00_R50/complete.json',NEXT_OUT/'B00_complete.json')
shutil.copy2(B00_ROOT/'split_manifest.csv',NEXT_OUT/'split_manifest.csv')
print('B00 validation:',base_scores)

# Dùng cho mọi ứng viên, ghép với B00 theo mã ảnh và nhãn.
def next_report(predictions,folder):
    folder=Path(folder);folder.mkdir(parents=True,exist_ok=True)
    assert predictions.image.is_unique and set(predictions.image)==set(truth.image)
    check=predictions.merge(truth,on='image',suffixes=('','_manifest'),validate='one_to_one')
    for c in ['group','file_sha256','true_label']:assert check[c].eq(check[c+'_manifest']).all()
    y=predictions.true_label.map(label_to_id);p=predictions.pred_label.map(label_to_id)
    assert y.notna().all() and p.notna().all()
    scores=metrics(y,p)
    predictions.to_csv(folder/'val_predictions.csv',index=False)
    dump_json(folder/'val_metrics.json',scores)
    report=classification_report(y,p,labels=range(8),target_names=LABELS,output_dict=True,zero_division=0)
    pd.DataFrame(report).T.to_csv(folder/'val_per_class.csv')
    cm=confusion_matrix(y,p,labels=range(8));pd.DataFrame(cm,index=LABELS,columns=LABELS).to_csv(folder/'val_confusion.csv')
    fig,ax=plt.subplots(figsize=(7,6));ax.imshow(cm,cmap='Blues')
    ax.set(xticks=range(8),yticks=range(8),xticklabels=LABELS,yticklabels=LABELS,xlabel='Predicted',ylabel='True')
    for i in range(8):
        for j in range(8):ax.text(j,i,str(cm[i,j]),ha='center',va='center',color='white' if cm[i,j]>cm.max()/2 else 'black')
    fig.tight_layout();fig.savefig(folder/'val_confusion.png',dpi=120);plt.close(fig)
    paired=base_predictions.merge(predictions,on='image',suffixes=('_B00','_candidate'),validate='one_to_one')
    b=paired.true_label_B00.eq(paired.pred_label_B00);a=paired.true_label_candidate.eq(paired.pred_label_candidate)
    paired['transition']=np.select([b&a,b&~a,~b&a],['right_to_right','right_to_wrong','wrong_to_right'],default='wrong_to_wrong')
    paired.to_csv(folder/'paired_validation.csv',index=False)
    pd.crosstab(paired.true_label_B00,paired.transition).reindex(LABELS,fill_value=0).to_csv(folder/'transitions_by_class.csv')
    return scores

B00 validation: {'accuracy': 0.7376920047262702, 'macro_f1': 0.5521146626587701, 'balanced_accuracy': 0.5403517144351776}


## 11. T02 — Chuẩn bị hai gói augmentation
Giữ B00 và split đã khóa. T02A lật ngang/dọc; T02B chọn đều tám phép đối xứng.
Chỉ train được augmentation; validation giữ resize/normalize B00. Xuất preview từ train để kiểm tra trước khi train.

In [11]:
# G1 — chạy sau mục 10 (S2) của notebook hiện tại.
from IPython.display import display
import gc

T02_ROOT = Path('/kaggle/working/ISIC2019_R50_T02_AB_v1')
T02_RESTORE = None  # Nếu đổi session: thư mục T02_AB_v1 FULL đã gắn Input.
METHODS_T02 = ('T02A_HV_R50', 'T02B_D4_R50')
assert RUN_TEST is False and A1_REVIEWED is False
assert reference['split_sha'] == b00_config['split_sha']
assert (EPOCHS, PATIENCE, SEED, IMG_SIZE, BATCH_SIZE, NUM_WORKERS) == (16, 4, 42, 224, 32, 2)
assert isinstance(criterion, nn.CrossEntropyLoss) and criterion.weight is None
assert criterion.label_smoothing == 0 and criterion.reduction == 'mean'

if T02_RESTORE is not None:
    source = Path(T02_RESTORE)
    assert source.is_dir() and source.resolve() != T02_ROOT.resolve()
    assert not T02_ROOT.exists(), 'Đích đã có dữ liệu: không ghi đè. Dùng session sạch hoặc bỏ RESTORE.'
    assert json.loads((source/'reference.json').read_text()) == reference
    shutil.copytree(source, T02_ROOT)
T02_ROOT.mkdir(parents=True, exist_ok=True)
next_lock(T02_ROOT/'reference.json', reference)
shutil.copy2(B00_ROOT/'split_manifest.csv', T02_ROOT/'split_manifest.csv')
shutil.copy2(B00_ROOT/'protocol_lock.json', T02_ROOT/'B00_protocol_lock.json')
shutil.copy2(B00_ROOT/'B00_R50/complete.json', T02_ROOT/'B00_complete.json')
base_predictions.to_csv(T02_ROOT/'B00_val_predictions.csv', index=False)

# Gói A: 4 phép, tương đương hai Bernoulli H/V độc lập p=0.5.
# Gói B: 8 phép D4 đồng xác suất, không phải xuất 8 ảnh cho mỗi mẫu.
def t02_geometry(rgb, method, code):
    assert rgb.shape == (IMG_SIZE, IMG_SIZE, 3)
    if method == 'T02A_HV_R50':
        assert code in range(4)
        if code & 1: rgb = np.flip(rgb, axis=1)
        if code & 2: rgb = np.flip(rgb, axis=0)
    elif method == 'T02B_D4_R50':
        assert code in range(8)
        rgb = np.rot90(rgb, k=code % 4, axes=(0, 1))
        if code >= 4: rgb = np.flip(rgb, axis=1)
    else:
        raise ValueError(method)
    return np.ascontiguousarray(rgb)

class T02Dataset(Dataset):
    def __init__(self, data, method, train=False):
        assert method in METHODS_T02
        self.data = data.reset_index(drop=True)
        self.method, self.train = method, train
    def __len__(self):
        return len(self.data)
    def __getitem__(self, index):
        row = self.data.iloc[index]
        with Image.open(row.path) as im:
            image = im.convert('RGB').resize((IMG_SIZE, IMG_SIZE), Image.Resampling.BILINEAR)
        if self.train:
            if self.method == 'T02A_HV_R50':
                code = int(torch.rand(()).item() < 0.5) + 2*int(torch.rand(()).item() < 0.5)
            else:
                code = int(torch.randint(8, ()).item())
            image = Image.fromarray(t02_geometry(np.asarray(image), self.method, code))
        return normalize(image), int(row.label)

def t02_loader(data, method, train=False):
    generator = torch.Generator().manual_seed(SEED)
    return DataLoader(T02Dataset(data, method, train), batch_size=BATCH_SIZE,
        shuffle=train, num_workers=NUM_WORKERS, pin_memory=True,
        worker_init_fn=seed_worker, generator=generator, drop_last=False)

def t02_preflight():
    # Kiểm chức năng B00 vẫn nguyên, trước khi chuyển loader tạm thời.
    src = '\n'.join(inspect.getsource(f) for f in [reset_seed, SkinDataset.__init__,
        SkinDataset.__getitem__, SkinDataset.__len__, seed_worker, make_loader,
        build_model, metrics, evaluate, export_evaluation, train_experiment])
    assert hashlib.sha256(src.encode()).hexdigest() == reference['training_source_sha']
    assert sha_file(B00_ROOT/'B00_R50/best_resnet50.pt') == reference['checkpoint_sha']
    assert DEVICE.type == 'cuda', 'Bật GPU T4.'
    assert torch.cuda.get_device_name(0) == b00_config['environment']['gpu']
    for key, value in dict(labels=LABELS, seed=SEED, size=IMG_SIZE, batch=BATCH_SIZE,
        epochs=EPOCHS, patience=PATIENCE, workers=NUM_WORKERS, lr=LR,
        weight_decay=WEIGHT_DECAY).items():
        assert value == b00_config[key], key
    # Kiểm split hiện trong RAM, không chỉ dựa vào số lượng ảnh.
    locked = pd.read_csv(B00_ROOT/'split_manifest.csv', keep_default_na=False)
    cols = ['image','label','label_name','group','file_sha256','split']
    for split in ['train','val','test']:
        pd.testing.assert_frame_equal(
            parts[split][cols].sort_values('image').reset_index(drop=True).astype(str),
            locked.loc[locked.split.eq(split), cols].sort_values('image').reset_index(drop=True).astype(str))
    # Xác nhận tensor validation mới giống pipeline B00, không augmentation.
    sample = parts['val'].groupby('label', group_keys=False).head(2)
    original = SkinDataset(sample, 'B00_R50')
    for method in METHODS_T02:
        candidate = T02Dataset(sample, method, train=False)
        for i in range(len(sample)):
            x, y = original[i]; z, t = candidate[i]
            assert y == t and torch.equal(x, z), 'Validation khác B00.'
    reset_seed(); probe = build_model(); h = hashlib.sha256()
    for key, value in probe.state_dict().items():
        h.update(key.encode()); h.update(value.detach().cpu().numpy().tobytes())
    assert h.hexdigest() == b00_result['initial_sha'], 'Khởi tạo khác B00.'
    del probe; gc.collect(); torch.cuda.empty_cache()

# Preview 1 ảnh TRAIN mỗi lớp, đủ mọi phép; không đọc ảnh test.
preview = parts['train'].sort_values('image').groupby('label', group_keys=False).head(1).sort_values('label')
assert len(preview) == 8
preview[['image','label_name','group']].to_csv(T02_ROOT/'preview_train_ids.csv', index=False)
for method in METHODS_T02:
    n = 4 if method == 'T02A_HV_R50' else 8
    fig, axes = plt.subplots(8, n+1, figsize=(2*(n+1), 16))
    for row_index, (_, row) in enumerate(preview.iterrows()):
        with Image.open(row.path) as im:
            rgb = np.asarray(im.convert('RGB').resize((IMG_SIZE, IMG_SIZE), Image.Resampling.BILINEAR))
        axes[row_index, 0].imshow(rgb)
        axes[row_index, 0].set_title(f'{row.label_name}\n{row.image}\nB00 resize', fontsize=7)
        # Kiểm toàn bộ bộ ba RGB được bảo toàn, không chỉ histogram từng kênh.
        packed = rgb.astype(np.uint32)
        expected = np.sort(((packed[:,:,0]<<16)+(packed[:,:,1]<<8)+packed[:,:,2]).ravel())
        for code in range(n):
            changed = t02_geometry(rgb, method, code)
            p = changed.astype(np.uint32)
            actual = np.sort(((p[:,:,0]<<16)+(p[:,:,1]<<8)+p[:,:,2]).ravel())
            assert np.array_equal(actual, expected)
            axes[row_index, code+1].imshow(changed)
            axes[row_index, code+1].set_title(f'code={code}', fontsize=8)
    for ax in axes.ravel(): ax.axis('off')
    fig.tight_layout(); fig.savefig(T02_ROOT/f'{method}_preview.png', dpi=100); plt.close(fig)
print('Đã tạo preview trong Output:', T02_ROOT)
print('Mở hai PNG trước khi chạy G2/G3. G1 chưa train; mỗi mẫu vẫn chỉ xuất một ảnh/lần lấy dữ liệu.')

Đã tạo preview trong Output: /kaggle/working/ISIC2019_R50_T02_AB_v1
Mở hai PNG trước khi chạy G2/G3. G1 chưa train; mỗi mẫu vẫn chỉ xuất một ảnh/lần lấy dữ liệu.


In [12]:
# Khởi tạo khóa baseline trước khi chạy các nhánh T02.
assert sha_file(B00_ROOT / 'protocol_lock.json') == b00_result['lock_sha']
LOCK_SHA = b00_result['lock_sha']

print("Đã khởi tạo LOCK_SHA của B00.")

Đã khởi tạo LOCK_SHA của B00.


## 12. T02A — Lật ngang/dọc
Train ResNet50 từ ImageNet với HFlip=0,5 và VFlip=0,5, CE thường.
Giữ ngân sách 16 epoch, patience4 và checkpoint theo validation macro-F1. Lưu riêng, không ghi đè B00.

In [13]:
# G2 — train T02A; dùng lại train_experiment nguyên bản qua loader riêng.
def run_t02(method):
    assert method in METHODS_T02
    t02_preflight()
    branch = T02_ROOT/method
    branch.mkdir(exist_ok=True)
    source = '\n\n'.join(inspect.getsource(f) for f in
        [t02_geometry, T02Dataset.__init__, T02Dataset.__len__,
         T02Dataset.__getitem__, t02_loader, t02_preflight, run_t02])
    spec = {'T02A_HV_R50': 'independent HFlip(p=0.5), VFlip(p=0.5)',
            'T02B_D4_R50': 'uniform code 0..7: rot90(k=code%4), then HFlip if code>=4'}
    config = dict(method=method, reference=reference, baseline_config=b00_config,
        candidate_augmentation=spec[method], augmentation_stage='train only, after RGB resize, before normalization',
        loss='CrossEntropyLoss(weight=None,label_smoothing=0)',
        validation='resize+normalize B00, single view, NO TTA',
        train_from='ImageNet V1; same initial state as B00; NOT B00 checkpoint',
        source_sha=hashlib.sha256(source.encode()).hexdigest(), environment=env)
    next_lock(branch/'candidate_protocol_lock.json', config)
    (branch/'candidate_source.py').write_text(source, encoding='utf-8')
    # Các hàm train/eval gốc tra make_loader và criterion qua globals.
    keys = ['OUT','LOCK_SHA','resume','make_loader','criterion']
    old = {key: globals()[key] for key in keys}
    try:
        globals().update(OUT=branch, LOCK_SHA=sha_file(branch/'candidate_protocol_lock.json'),
            resume=None, make_loader=t02_loader, criterion=nn.CrossEntropyLoss())
        result = train_experiment(method)
    finally:
        globals().update(old)
        gc.collect(); torch.cuda.empty_cache()
    assert result['initial_sha'] == b00_result['initial_sha']
    folder = branch/method
    scores = next_report(pd.read_csv(folder/'val_predictions.csv'), branch/'comparison')
    for key, value in scores.items():
        assert np.isclose(value, result[key], rtol=0, atol=1e-10)
    dump_json(branch/'budget_note.json', dict(
        best_epoch=result['best_epoch'], completed_epochs=result['completed_epochs'],
        best_in_last_two_budget_epochs=result['best_epoch'] >= EPOCHS-1,
        interpretation='Late best suggests checking budget limits; does not justify increasing epochs for this branch alone.'))
    print(method, result)
    return result

result_t02a = run_t02('T02A_HV_R50')

Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 196MB/s]


T02A_HV_R50 epoch 01: train=0.8620, val=0.7957, F1=0.4760
T02A_HV_R50 epoch 02: train=0.6642, val=0.7559, F1=0.5288
T02A_HV_R50 epoch 03: train=0.5598, val=0.7845, F1=0.5186
T02A_HV_R50 epoch 04: train=0.4678, val=0.7515, F1=0.5475
T02A_HV_R50 epoch 05: train=0.3842, val=0.8290, F1=0.5685
T02A_HV_R50 epoch 06: train=0.3158, val=0.7803, F1=0.5788
T02A_HV_R50 epoch 07: train=0.2477, val=0.8353, F1=0.5853
T02A_HV_R50 epoch 08: train=0.1851, val=0.8916, F1=0.5655
T02A_HV_R50 epoch 09: train=0.1426, val=0.9569, F1=0.5788
T02A_HV_R50 epoch 10: train=0.1015, val=0.9481, F1=0.5818
T02A_HV_R50 epoch 11: train=0.0665, val=1.0350, F1=0.5837
T02A_HV_R50 {'method': 'T02A_HV_R50', 'lock_sha': 'acb7134839f149ed7f4b6cdc234e40bb7f058d376c077bae0623358651596a35', 'initial_sha': '37a03a16d86fda7f927000366760327e605b16fc8b4cb6a565a4ce763fe9003c', 'checkpoint_sha': 'c2b42cfe80c67c3a26125c42a1941bcf79d4aea3fe81629ba0aaa2027a03cdfd', 'best_epoch': 7, 'completed_epochs': 11, 'seconds': 2360.711708585, 'accura

## 13. T02B — Tám phép đối xứng hình vuông
Train độc lập từ cùng ImageNet initialization, chọn đều một trong tám phép D4 trên mỗi ảnh train.
Không tiếp tục từ T02A; validation không TTA và không xoay/lật.

In [14]:
# G3 — lượt độc lập, khởi tạo lại ImageNet, không dùng checkpoint T02A.
result_t02b = run_t02('T02B_D4_R50')

T02B_D4_R50 epoch 01: train=0.8687, val=0.7835, F1=0.4492
T02B_D4_R50 epoch 02: train=0.6839, val=0.7540, F1=0.5538
T02B_D4_R50 epoch 03: train=0.6008, val=0.7765, F1=0.5439
T02B_D4_R50 epoch 04: train=0.5250, val=0.7282, F1=0.5397
T02B_D4_R50 epoch 05: train=0.4576, val=0.7511, F1=0.5560
T02B_D4_R50 epoch 06: train=0.3902, val=0.7720, F1=0.5569
T02B_D4_R50 epoch 07: train=0.3328, val=0.8238, F1=0.5412
T02B_D4_R50 epoch 08: train=0.2745, val=0.7990, F1=0.5707
T02B_D4_R50 epoch 09: train=0.2245, val=0.8630, F1=0.5697
T02B_D4_R50 epoch 10: train=0.1724, val=0.8756, F1=0.5575
T02B_D4_R50 epoch 11: train=0.1308, val=0.9460, F1=0.5573
T02B_D4_R50 epoch 12: train=0.1024, val=0.9170, F1=0.5828
T02B_D4_R50 epoch 13: train=0.0713, val=0.9388, F1=0.5852
T02B_D4_R50 epoch 14: train=0.0646, val=0.9233, F1=0.6030
T02B_D4_R50 epoch 15: train=0.0522, val=0.9528, F1=0.5863
T02B_D4_R50 epoch 16: train=0.0456, val=0.9628, F1=0.5834
T02B_D4_R50 {'method': 'T02B_D4_R50', 'lock_sha': '47eacd88da997d3df482a

## 14. So sánh B00–T02A–T02B trên validation
Đối chiếu dự đoán theo mã ảnh, nhãn, nhóm và hash. Báo macro-F1, recall và chuyển đúng/sai.
Paired bootstrap lấy lại nhóm tổn thương trong từng lớp, dùng cùng mẫu cho mọi mô hình.
Khoảng tin cậy chỉ mô tả bất định trên validation hiện tại, không thay thế nhiều seed hoặc test độc lập.

In [15]:
# G4 — CPU: bảng so sánh + paired cluster bootstrap; không train và không đọc ảnh test.
def t02_bootstrap(predictions, n_boot=2000, seed=20261007):
    names = list(predictions)
    base = predictions[names[0]].sort_values('image').reset_index(drop=True)
    required = ['image','group','file_sha256','true_label']
    assert base.image.is_unique and base.groupby('group').true_label.nunique().max() == 1
    aligned = []
    for name in names:
        df = predictions[name].sort_values('image').reset_index(drop=True)
        assert df.image.is_unique
        pd.testing.assert_frame_equal(base[required].astype(str), df[required].astype(str))
        ids = df.pred_label.map(label_to_id)
        assert ids.notna().all()
        aligned.append(ids.to_numpy(dtype=int))
    y = base.true_label.map(label_to_id).to_numpy(dtype=int)
    codes, groups = pd.factorize(base.group, sort=True)
    group_y = np.array([y[np.flatnonzero(codes == g)[0]] for g in range(len(groups))])
    strata = [np.flatnonzero(group_y == k) for k in range(8)]
    assert all(len(s) > 0 for s in strata)
    # Mỗi group giữ mọi ảnh của nó. Chỉ lấy lại group, không lấy lại từng ảnh độc lập.
    counts = np.zeros((len(names), len(groups), 8), dtype=np.int64)
    for model_id, pred in enumerate(aligned):
        np.add.at(counts[model_id], (codes, pred), 1)
    def macro(cm):
        tp = np.diagonal(cm, axis1=-2, axis2=-1)
        denom = cm.sum(axis=-1) + cm.sum(axis=-2)
        return np.divide(2*tp, denom, out=np.zeros_like(tp, dtype=float), where=denom>0).mean(axis=-1)
    observed_cm = np.stack([confusion_matrix(y, pred, labels=range(8)) for pred in aligned])
    observed = macro(observed_cm)
    rng = np.random.default_rng(seed)
    draws = np.empty((n_boot, len(names)))
    for b in range(n_boot):
        cm = np.zeros((len(names), 8, 8), dtype=np.int64)
        for label, ids in enumerate(strata):
            chosen = rng.choice(ids, size=len(ids), replace=True)
            cm[:, label, :] = counts[:, chosen, :].sum(axis=1)
        draws[b] = macro(cm)
    pairs = [(i,0) for i in range(1,len(names))]
    if len(names) == 3: pairs.append((2,1))  # B so với A, bổ sung mô tả.
    records = []
    for i,j in pairs:
        delta = 100*(draws[:,i]-draws[:,j])
        lo,hi = np.quantile(delta, [0.025,0.975])
        records.append(dict(candidate=names[i], reference=names[j],
            delta_macro_f1_pp=100*(observed[i]-observed[j]),
            ci95_low_pp=float(lo), ci95_high_pp=float(hi), n_boot=n_boot,
            contains_zero=bool(lo<=0<=hi)))
    group_support = pd.DataFrame(dict(label=LABELS, n_groups=[len(s) for s in strata]))
    return pd.DataFrame(records), pd.DataFrame(draws,columns=names), group_support

def export_t02_reports():
    predictions = {'B00_R50_CE': base_predictions}
    rows = [dict(method='B00_R50_CE', **base_scores,
        best_epoch=b00_result['best_epoch'], completed_epochs=b00_result['completed_epochs'])]
    for method in METHODS_T02:
        branch = T02_ROOT/method
        folder = branch/method
        if not (folder/'complete.json').exists():
            print('Chưa hoàn thành:', method)
            continue
        result = json.loads((folder/'complete.json').read_text())
        assert result['lock_sha'] == sha_file(branch/'candidate_protocol_lock.json')
        assert result['initial_sha'] == b00_result['initial_sha']
        assert sha_file(folder/'best_resnet50.pt') == result['checkpoint_sha']
        pred = pd.read_csv(folder/'val_predictions.csv')
        scores = next_report(pred, branch/'comparison')
        predictions[method] = pred
        rows.append(dict(method=method, **scores, best_epoch=result['best_epoch'],
                         completed_epochs=result['completed_epochs']))
    table = pd.DataFrame(rows)
    table['delta_macro_f1_pp'] = 100*(table.macro_f1-base_scores['macro_f1'])
    table.to_csv(T02_ROOT/'comparison_validation.csv', index=False)
    display(table)
    if len(predictions) > 1:
        ci, draws, support = t02_bootstrap(predictions)
        ci.to_csv(T02_ROOT/'bootstrap_delta_macro_f1.csv', index=False)
        draws.to_csv(T02_ROOT/'bootstrap_macro_f1_draws.csv', index=False)
        support.to_csv(T02_ROOT/'bootstrap_group_support.csv', index=False)
        display(ci)
    dump_json(T02_ROOT/'analysis_protocol.json', dict(
        bootstrap='paired class-stratified cluster bootstrap, 2000 draws, seed 20261007',
        unit='group from frozen manifest; all images in sampled group retained',
        interval='pointwise percentile 95%, not multiple-comparison-adjusted',
        limits='Conditional on this validation set and fitted checkpoints; no training-seed uncertainty or correction for checkpoint/model selection; missing lesion IDs remain a limitation.',
        test_used=False, tta_used=False))
    (T02_ROOT/'analysis_source.py').write_text(
        inspect.getsource(t02_bootstrap)+'\n'+inspect.getsource(export_t02_reports), encoding='utf-8')
    archive = T02_ROOT.parent/(T02_ROOT.name+'_reports.zip')
    with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
        for p in sorted(T02_ROOT.rglob('*')):
            if p.is_file() and p.suffix not in ['.pt','.npy','.joblib']:
                z.write(p, str(p.relative_to(T02_ROOT)))
    print('Gửi ZIP + notebook đã chạy:', archive)
    print('Giữ FULL Output để tái sử dụng checkpoint. ZIP reports không chứa .pt.')
    print('CI chỉ mô tả bất định lấy mẫu validation; không chứng minh thắng chắc hoặc thay repeated seeds.')
    return table

t02_comparison = export_t02_reports()

,method,accuracy,macro_f1,balanced_accuracy,best_epoch,completed_epochs,delta_macro_f1_pp
0,B00_R50_CE,0.737692,0.552115,0.540352,14,16,0.000000
1,T02A_HV_R50,0.744781,0.585276,0.571724,7,11,3.316173
2,T02B_D4_R50,0.769988,0.603002,0.586105,14,16,5.088718


,candidate,reference,delta_macro_f1_pp,ci95_low_pp,ci95_high_pp,n_boot,contains_zero
0,T02A_HV_R50,B00_R50_CE,3.316173,-1.129527,8.147134,2000,True
1,T02B_D4_R50,B00_R50_CE,5.088718,0.173874,9.388386,2000,False
2,T02B_D4_R50,T02A_HV_R50,1.772545,-2.865837,5.869755,2000,True


Gửi ZIP + notebook đã chạy: /kaggle/working/ISIC2019_R50_T02_AB_v1_reports.zip
Giữ FULL Output để tái sử dụng checkpoint. ZIP reports không chứa .pt.
CI chỉ mô tả bất định lấy mẫu validation; không chứng minh thắng chắc hoặc thay repeated seeds.
